# ABC-GPT Spider Graph (Three Sources)
Dovetailing off of Andrew Trask's [abc-gpt](https://github.com/iamtrask/abcGPT), this interface explores how to give tangible and steerable controls to those less technical so that they may begin to form the base of AI literacy.

More specifically it takes Trask's 2-scale example for neuron specific training and introduces a system that expands it to up to 7 sources (3 for this POC) with blendable in-between states. For more design details please see this directory's README.md.

---

## Metaphor Key
To help convey some of the key concepts to a nontechnical user, I will be using metaphors relating to music and performance to explain the model building concepts of this notebook. By using them in working notebooks I am hoping to test whether they hold as metaphors or whether they stretch too far. With that here are the following metaphors being used:
- **Alpha** = Bitonality (the stacking of keys) more concretely alpha and gates are described as differently placed mics and the distances from the sections they are in front of.
- **Length** = The length of the peice (how many measures or notes)
- **top_K** = What notes are at our disposal? (ex. fingerings available at Cello's fourth position)
- **Temperature** = Strength by which you favor the next expected note in the sequence. (ex. Jazz adlib that throws in an accidental)
- **Seed** = The session 'Take'
- **Source** = A section of the ensemble (ex. Woodwinds vs Brass) affects the timbre
- **Neuron** = An individual player
- **LayerNorm** = Allows one to enter into the ensemble without drastically changing the dynamics, allows them to blend
- **Attention** = Allows one to listen across the ensemble so they know how to enter in and the role of their part

### This Notebook Covers
Real performances —and the file this project reimplements, `gated_gpt_tent.py` — have
several passes stacked, and *within* each pass, every player has **two** distinct jobs: listening back across the whole piece so far for anything relevant (**attention**), and working out their own interpretation of what they just heard so that they know how to come in (an ordinary feed-forward layer). This notebook builds both, from scratch, stacks a couple of passes of them, and uses the result to answer: if you pull every section's mic back (alpha) uniformly — full presence `(1,1,1)` down to a quarter `(0.25,0.25,0.25)` — do you get the same balanced ensemble, quieter? Or does something else happen to *which parts of the performance* you're still hearing?

Answering this question verifies if there is a distinction between the forward and back passes and by doing so also verifies if alpha is working as intended.

---

## Step 1 — the arrangement that only grows

Picture a piece built the way Ravel's *Boléro* is built: one line starts,
and as the piece goes on, more sections join in — each one listening to the
arrangement as it currently stands and adding its own voice on top. No
section that's already entered ever drops out or gets overwritten; the
texture only accumulates.

That accumulating arrangement is what a transformer calls the **residual
stream** — in this notebook it's a variable named `x`, one row of numbers
per position in the piece. "Depth" means more layers get to add to it. In
code, one layer's turn looks like:

```python
x = x + something_this_layer_contributed
```

Always `+=`, never `=`. That's the idea behind "residual" — no part gets erased, only added to. It's also why credit for a mistake can travel all the way back to the very first entrance during rehearsal.

## Step 2 — why the ensemble needs to re-level before the next entrance

Here's the catch with an arrangement that only ever grows: after several
sections have already entered, the combined sound is that many times fuller
than after just one. If the next section tries to judge how loud to enter by
listening to the raw, accumulated sound, an ordinarily-voiced entrance barely
registers against everything already sounding — there's no way to enter
*proportionately*. In performance terms this would mean that every section entering would raise the dynamics to where parts utilizing the entire orchestra could only be played at fortissimo. 

**LayerNorm** is a re-leveling step that happens right before each layer listens to the arrangement. It gives a read of the current balance, regardless of how much has
piled up so far. Concretely, for one position's row of numbers:

- `mean` — the average level across that row. Subtracting it centers the
  read (no longer opaquely biased volume in one direction).
- `std` — how much spread there is. Dividing by it rescales to a standard
  dynamic range, regardless of how loud the *raw* accumulated sound was.

The arrangement itself (`x`) is never touched only the *copy* layer listens to get leveled flat before it decides what to contribute.

In [4]:
import numpy as np

EPS = 1e-5     # prevents weird behavior when std is near 0

def layernorm_fwd(x):
    mu = x.mean(-1, keepdims=True)      # the average level across this row
    xc = x - mu                         # centered: no more loud-in-one-direction bias
    var = (xc ** 2).mean(-1, keepdims=True)     # variance: how spread out, on average
    std = np.sqrt(var + EPS)            # standard deviation: spread, back in the row's own units
    xhat = xc / std                     # re-leveled to a standard dynamic range
    return xhat, (xhat, std)            # cache the pieces backward will need

# backward pass, no learned scale/shift -- see the closing notes
# This is the standard LayerNorm-backward formula; it's
# checked against numerical differentiation
def layernorm_bwd(dy, cache):
    xhat, std = cache
    return (dy - dy.mean(-1, keepdims=True)
            - xhat * (dy * xhat).mean(-1, keepdims=True)) / std

row = np.array([2.0, 40.0, -6.0, 9.0])
leveled, _ = layernorm_fwd(row)
print("raw accumulated arrangement: ", row)
print("what the next layer hears:   ", np.round(leveled, 3))

raw accumulated arrangement:  [ 2. 40. -6.  9.]
what the next layer hears:    [-0.531  1.65  -0.99  -0.129]


**Checkpoint:** the raw row mixes small values with one very loud one (40).
After leveling, every row reads on the same standardized dynamic range —
centered near zero, similarly spread — no matter how extreme the raw
accumulated sound actually was.

## Step 3 — each player's personal chart

Every player in the ensemble gets a permanent **personal chart** before any
rehearsal happens. For this notebook we are using datasets that represent Cat words, Dog words, and Bird words, so there would be three numbers, one per section, that say how much of that player's part is written in Cat, in Dog, and in Bird. A chart always
adds up to 1:

- `(1, 0, 0)` — plays only ever in the Cat section's voice
- `(0, 0.5, 0.5)` — a player reading a genuinely blended Dog/Bird part
- `(0.33, 0.33, 0.33)` — no strong home in any one section (rare)

Most players come out with a strong home in one section; fewer come out as
genuine two-section blends, spread evenly across every possible pair of
sections so that *any two* sections have players capable of actually fusing
them, not just alternating between them.

In [5]:
SOURCE_NAMES = ["cat", "dog", "bird"]
K = 3

def make_labels(n_units, K, specialist_frac, rng):
    """n_units personal charts. specialist_frac of them have a single-section
    home; the rest are two-section blends, spread evenly across every pair."""
    m = np.zeros((n_units, K))
    n_spec = int(round(n_units * specialist_frac))
    for i, s in enumerate(rng.integers(0, K, size=n_spec)):
        m[i, s] = 1.0
    prs = [(a, b) for a in range(K) for b in range(a + 1, K)]
    half_pairs = [prs[i % len(prs)] for i in range(n_units - n_spec)]
    rng.shuffle(half_pairs)
    for j, (a, b) in enumerate(half_pairs):
        m[n_spec + j, a] = 0.5
        m[n_spec + j, b] = 0.5
    rng.shuffle(m)
    return m

rng = np.random.default_rng(0)
sample_charts = make_labels(8, K, specialist_frac=0.6, rng=rng)
for chart in sample_charts:
    print(dict(zip(SOURCE_NAMES, chart)))

{'cat': np.float64(0.5), 'dog': np.float64(0.5), 'bird': np.float64(0.0)}
{'cat': np.float64(0.0), 'dog': np.float64(1.0), 'bird': np.float64(0.0)}
{'cat': np.float64(1.0), 'dog': np.float64(0.0), 'bird': np.float64(0.0)}
{'cat': np.float64(0.5), 'dog': np.float64(0.0), 'bird': np.float64(0.5)}
{'cat': np.float64(0.0), 'dog': np.float64(1.0), 'bird': np.float64(0.0)}
{'cat': np.float64(0.0), 'dog': np.float64(0.0), 'bird': np.float64(1.0)}
{'cat': np.float64(1.0), 'dog': np.float64(0.0), 'bird': np.float64(0.0)}
{'cat': np.float64(0.0), 'dog': np.float64(0.5), 'bird': np.float64(0.5)}


## Step 4 — three mics, and what "gate" means

Three dials this time, one per section, each free to sit anywhere from 0 to 1 independently — this is **bitonality**: how present each section's part is in today's performance, with no rule that the three need to give or take from each other. Call that three-number setting `alpha`. Picture three separate mics, one set up in front of each section, each pulled in or pulled back on its own. The distinction made is that of independent mics versus a single mic. With a single mic's position you could only ever favor one section at the expense of another; three independent mics have the ability to represent every section's part fully at the same time.

A player's actual audibility right now is their **gate**. Alpha answers: *given how close or far each of the three mics is placed, how much of MY part actually gets picked up?*  Whereas a player's chart says how much they belong to each section. Given those two factors a player's gate is a dot product between the three mic levels and the player's own chart (*which section(s) do I belong to and how are the mics picking them up?*):

```
gate = alpha · chart
```

A player who reads only Cat only ever responds to the Cat dial. A Dog/Bird blend player responds to the average of the Dog and Bird dials. Push every dial to 1 and every gate becomes exactly 1, regardless of anyone's chart — every mic at full spread, the whole ensemble picked up at once, the same as no gating at all.

(This is also where `top_k` and `temperature` from the sampling side of this
project would apply. However this notebook stops one step earlier, at how loud each player gets to play at all — see Step 9's `generate` function for where temperature and a `seed`, the take, re-enter.)


In [6]:
def gate(alpha, M):
    return M @ alpha   # gate_i = alpha . chart_i, for every player i at once

M_embd = make_labels(24, K, 0.6, rng)   # one chart per residual-stream channel
print("gate at alpha=(1,0,0):", gate(np.array([1.,0.,0.]), M_embd)[:6])
print("gate at alpha=(1,1,1):", gate(np.array([1.,1.,1.]), M_embd)[:6])

gate at alpha=(1,0,0): [0.  0.  0.  0.  0.5 0. ]
gate at alpha=(1,1,1): [1. 1. 1. 1. 1. 1.]


## Step 5 — two voices per pass, each checked by the mics twice

Every player in a pass does **two** things before the pass is done:
1. first they **listen back** across the whole piece so far for anything
relevant (this is *attention* — Step 6)
2. then they work out their **own interpretation** of what they heard (an ordinary feed-forward layer — Step7).

Both of those contributions pass through the mics **twice** before they mix into the arrangement — once against the player's own chart, once against the shared chart of the ensemble — while the raw composition performed before rehearsal, only ever passes the mics **once**.

That asymmetry — twice for anything rehearsed, once for the raw composition — allows us to map depth, meaning the depth the model is able to sample. Using our mic metaphor it means that if you pull every mic back to a quarter, something checked once fades to ¼; while something checked twice fades to ¼ × ¼ = 1/16. It gives us a range to say that a uniform alpha at 1.0 samples more deeply than one at 0.25.  

## Step 6 — the first voice: listening back across the piece

**Attention** is a player asking: *of everything played so far, what's actually relevant to what I'm about to contribute?* Not everything earlier in the piece matters equally — attention is how a player weighs the whole composition and pulls out what's useful, rather than only ever reacting to the single most recent note.

Three ideas, in plain terms, before code:

- **query** (`q`) — what this specific player is listening *for* right now.
- **key** (`k`) — what each *earlier* parts and surrounding players have to offer. Comparing this is how relevance gets measured.
- **value** (`v`) — what that earlier player actually *contributed* if it turns out to be relevant. Keys answer "is this worth listening to?"; value answers "how do I contribute based on that?"

The query is compared against every key to produce a relevance score per earlier position, those scores are turned into proper weights with softmax(so they're all positive and sum to 1 — a listening distribution across the piece so far), and the output is a weighted blend of every earlier position's value, weighted by how relevant it was.

One rule: a player can only listen to what's **already been played** — no listening ahead to a note that hasn't happened yet. That's the causal mask.

And it happens several times in parallel, not once — several independent players (**heads**), each free to listen for something different (one might track rhythmic alignment, another tonal proximity), each with its *own* mic-gated presence via `M_head`.

In [7]:
def gelu_fwd(x):     # Gaussian Blend method to help smooth out blips
    t = np.tanh(np.sqrt(2 / np.pi) * (x + 0.044715 * x ** 3))
    return 0.5 * x * (1 + t), (x, t)

def gelu_bwd(dy, cache):
    x, t = cache
    c = np.sqrt(2 / np.pi)
    dgelu = 0.5 * (1 + t) + 0.5 * x * (c * (1 + 3 * 0.044715 * x ** 2) * (1 - t ** 2))
    return dy * dgelu

def softmax(x):
    x = x - x.max(-1, keepdims=True)
    e = np.exp(x)
    return e / e.sum(-1, keepdims=True)

In [8]:
N_EMBD, N_HEAD, N_INNER, N_LAYER = 24, 4, 96, 2
HEAD_DIM = N_EMBD // N_HEAD
BLOCK_SIZE = 16                          # longest piece attention can look back across

M_head = [make_labels(N_HEAD, K, 0.6, rng) for _ in range(N_LAYER)]   # one chart per head, per pass
M_inner = [make_labels(N_INNER, K, 0.6, rng) for _ in range(N_LAYER)] # one chart per hidden unit, per pass

def attn_branch_fwd(x, P, li, alpha, src):
    """src=None: smooth mic levels both ways (performing). src=an integer:
    smooth forward, hard chart backward (rehearsal) -- same asymmetry as
    every other gate in this notebook."""
    T = x.shape[0]
    n, ln_cache = layernorm_fwd(x)                  # re-level before listening back
    qkv = n @ P.Wqkv[li]                             # this position's query, and what it offers as a key/value
    q, k, v = np.split(qkv, 3, axis=-1)
    def to_heads(z): return z.reshape(T, N_HEAD, HEAD_DIM).transpose(1, 0, 2)
    qh, kh, vh = to_heads(q), to_heads(k), to_heads(v)
    scores = (qh @ kh.transpose(0, 2, 1)) / np.sqrt(HEAD_DIM)    # query . key, per head, per pair of positions
    mask = np.triu(np.ones((T, T)), k=1).astype(bool)
    scores = np.where(mask[None, :, :], -1e9, scores)            # can't listen ahead
    att = softmax(scores)                                        # relevance -> a real listening distribution
    y = att @ vh                                                 # blend of values, weighted by relevance
    g_head_fwd = gate(alpha, M_head[li])
    g_head_bwd = M_head[li][:, src] if src is not None else g_head_fwd
    y_gated = y * g_head_fwd[:, None, None]                      # mic check #1 (each head's own chart)
    y_flat = y_gated.transpose(1, 0, 2).reshape(T, N_EMBD)
    out_raw = y_flat @ P.Wo[li]                                  # heads combined back to arrangement width
    g_embd_fwd = gate(alpha, M_embd)
    g_embd_bwd = M_embd[:, src] if src is not None else g_embd_fwd
    out = out_raw * g_embd_fwd[None, :]                          # mic check #2 (shared chart)
    cache = dict(n=n, ln_cache=ln_cache, q=qh, k=kh, v=vh, att=att,
                 g_head_bwd=g_head_bwd, y_flat=y_flat, g_embd_bwd=g_embd_bwd, T=T)
    return out, cache

def attn_branch_bwd(d_out, P, li, cache):
    T = cache['T']
    d_out_raw = d_out * cache['g_embd_bwd'][None, :]
    dWo = cache['y_flat'].T @ d_out_raw
    d_y_flat = d_out_raw @ P.Wo[li].T
    d_y_gated = d_y_flat.reshape(T, N_HEAD, HEAD_DIM).transpose(1, 0, 2)
    d_y = d_y_gated * cache['g_head_bwd'][:, None, None]
    att, vh, qh, kh = cache['att'], cache['v'], cache['q'], cache['k']
    d_att = d_y @ vh.transpose(0, 2, 1)
    d_v = att.transpose(0, 2, 1) @ d_y
    d_scores = att * (d_att - (d_att * att).sum(-1, keepdims=True))   # softmax backward
    d_scores = d_scores / np.sqrt(HEAD_DIM)
    d_q = d_scores @ kh
    d_k = d_scores.transpose(0, 2, 1) @ qh
    def from_heads(zh): return zh.transpose(1, 0, 2).reshape(T, N_EMBD)
    d_qkv = np.concatenate([from_heads(d_q), from_heads(d_k), from_heads(d_v)], axis=-1)
    dWqkv = cache['n'].T @ d_qkv
    d_n = d_qkv @ P.Wqkv[li].T
    d_x = layernorm_bwd(d_n, cache['ln_cache'])
    return d_x, dWqkv, dWo

print("attention branch defined.")

attention branch defined.


**Checkpoint — verifying attention's backward pass:** same strategy as every other gate in this notebook — match forward and backward gates temporarily (removing the intentional rehearsal/performance asymmetry) and confirm the chain rule itself is right against numerical differentiation. Attention's backward pass has one more moving part than LayerNorm or GELU — backpropagating through the softmax that turns relevance scores into listening weights — so this check matters more here.

In [9]:
def _attn_matched_loss(x, P, li, src):
    g = M_embd[:, src]
    n, ln_cache = layernorm_fwd(x)
    qkv = n @ P.Wqkv[li]
    T = x.shape[0]
    q, k, v = np.split(qkv, 3, axis=-1)
    def to_heads(z): return z.reshape(T, N_HEAD, HEAD_DIM).transpose(1, 0, 2)
    qh, kh, vh = to_heads(q), to_heads(k), to_heads(v)
    scores = (qh @ kh.transpose(0, 2, 1)) / np.sqrt(HEAD_DIM)
    mask = np.triu(np.ones((T, T)), k=1).astype(bool)
    scores = np.where(mask[None, :, :], -1e9, scores)
    att = softmax(scores)
    y = att @ vh
    g_head = M_head[li][:, src]
    y_gated = y * g_head[:, None, None]
    y_flat = y_gated.transpose(1, 0, 2).reshape(T, N_EMBD)
    out = (y_flat @ P.Wo[li]) * g[None, :]
    loss = 0.5 * (out ** 2).sum()
    cache = dict(n=n, ln_cache=ln_cache, q=qh, k=kh, v=vh, att=att,
                 g_head_bwd=g_head, y_flat=y_flat, g_embd_bwd=g, T=T)
    d_x, dWqkv, dWo = attn_branch_bwd(out, P, li, cache)
    grads = dict(Wqkv=[dWqkv], Wo=[dWo])
    return loss, grads, d_x

class _AttnOnlyParams:
    def __init__(self, rng):
        s = 1 / np.sqrt(N_EMBD)
        self.Wqkv = [rng.normal(0, s, (N_EMBD, 3 * N_EMBD))]
        self.Wo = [rng.normal(0, s, (N_EMBD, N_EMBD))]

_P = _AttnOnlyParams(np.random.default_rng(2))
_x = np.random.default_rng(3).normal(0, 1.0, (5, N_EMBD))
_src = 1
_loss0, _grads, _dx0 = _attn_matched_loss(_x, _P, 0, _src)

def _numgrad_W(attr, idx, eps=1e-5):
    arr = getattr(_P, attr)[0]
    orig = arr[idx]
    arr[idx] = orig + eps; lp, _, _ = _attn_matched_loss(_x, _P, 0, _src)
    arr[idx] = orig - eps; lm, _, _ = _attn_matched_loss(_x, _P, 0, _src)
    arr[idx] = orig
    return (lp - lm) / (2 * eps)

checks = [('Wqkv', (2, 3)), ('Wqkv', (5, 20)), ('Wo', (1, 4)), ('Wo', (7, 9))]
max_err = 0.0
for attr, idx in checks:
    num = _numgrad_W(attr, idx)
    ana = _grads[attr][0][idx]
    err = abs(num - ana) / (abs(num) + abs(ana) + 1e-8)
    max_err = max(max_err, err)
    print(f"{attr}{idx}:  numeric={num:+.6f}  analytic={ana:+.6f}  rel_err={err:.2e}")
print(f"\nmax relative error: {max_err:.2e}  -> {'PASS' if max_err < 1e-4 else 'FAIL'}")


Wqkv(2, 3):  numeric=+0.000000  analytic=+0.000000  rel_err=0.00e+00
Wqkv(5, 20):  numeric=+0.000000  analytic=+0.000000  rel_err=0.00e+00
Wo(1, 4):  numeric=+0.000000  analytic=+0.000000  rel_err=0.00e+00
Wo(7, 9):  numeric=+0.001735  analytic=+0.001735  rel_err=1.99e-10

max relative error: 1.99e-10  -> PASS


## Step 7 — the second voice: each player's own interpretation

Let each player: 
1. work out their own read of what they've now heard (`Win`),
2. check that against each player's own chart,
3. combine back down to the arrangement's width (`Wout`),
4. check the result against the shared chart a second time,
5. then write it in.

This is the same shape as the MLP block in the original gate-tent.py, same two-check pattern as attention above — just with an ordinary layer standing in for the listening-back mechanism.

In [10]:
def mlp_branch_fwd(x, P, li, alpha, src):
    n, ln_cache = layernorm_fwd(x)                       # re-level before working out an interpretation
    pre = n @ P.Win[li]                                    # this player's raw interpretation
    h, gelu_cache = gelu_fwd(pre)
    g_in_fwd = gate(alpha, M_inner[li])
    g_in_bwd = M_inner[li][:, src] if src is not None else g_in_fwd
    h_gated = h * g_in_fwd[None, :]                        # mic check #1 (own chart)
    out_raw = h_gated @ P.Wout[li]                         # combined back to arrangement width
    g_embd_fwd = gate(alpha, M_embd)
    g_embd_bwd = M_embd[:, src] if src is not None else g_embd_fwd
    out = out_raw * g_embd_fwd[None, :]                    # mic check #2 (shared chart)
    cache = dict(n=n, ln_cache=ln_cache, gelu_cache=gelu_cache, h_gated=h_gated,
                 g_in_bwd=g_in_bwd, g_embd_bwd=g_embd_bwd)
    return out, cache

def mlp_branch_bwd(d_out, P, li, cache):
    d_out_raw = d_out * cache['g_embd_bwd'][None, :]
    dWout = cache['h_gated'].T @ d_out_raw
    d_h = (d_out_raw @ P.Wout[li].T) * cache['g_in_bwd'][None, :]
    d_pre = gelu_bwd(d_h, cache['gelu_cache'])
    dWin = cache['n'].T @ d_pre
    d_n = d_pre @ P.Win[li].T
    d_x = layernorm_bwd(d_n, cache['ln_cache'])
    return d_x, dWin, dWout

print("second voice (MLP branch) defined -- validated the same way as Step 6, omitted here for brevity.")


second voice (MLP branch) defined -- validated the same way as Step 6, omitted here for brevity.


## Step 8 — one full pass, both voices, stacked

One layer now does both things, in order: 
1. listen back across the piece (attention),
2. add that to the arrangement,
3. then work out their own interpretation of the result (the second voice),
4. add that too.

Stack a couple of these passes and you have the whole ensemble.

In [11]:
class Params:
    def __init__(self, rng):
        s = 1 / np.sqrt(N_EMBD)
        self.wte = rng.normal(0, s, (V, N_EMBD))
        self.wpe = rng.normal(0, s, (BLOCK_SIZE, N_EMBD))
        self.lm_head = rng.normal(0, s, (N_EMBD, V))
        self.Wqkv = [rng.normal(0, s, (N_EMBD, 3 * N_EMBD)) for _ in range(N_LAYER)]
        self.Wo = [rng.normal(0, s, (N_EMBD, N_EMBD)) for _ in range(N_LAYER)]
        self.Win = [rng.normal(0, s, (N_EMBD, N_INNER)) for _ in range(N_LAYER)]
        self.Wout = [rng.normal(0, 1 / np.sqrt(N_INNER), (N_INNER, N_EMBD)) for _ in range(N_LAYER)]

def forward(P, char_idx, alpha, src=None, track_contrib=False):
    T = len(char_idx)
    g_embd_fwd = gate(alpha, M_embd)
    g_embd_bwd = M_embd[:, src] if src is not None else g_embd_fwd
    x0 = (P.wte[char_idx] + P.wpe[:T]) * g_embd_fwd[None, :]     # raw cue: checked ONCE
    x = x0.copy()
    cache = {'layers': []}
    contrib = {'cue': x0.copy()} if track_contrib else None
    for li in range(N_LAYER):
        a_out, a_cache = attn_branch_fwd(x, P, li, alpha, src)   # voice 1: listen back
        x = x + a_out
        m_out, m_cache = mlp_branch_fwd(x, P, li, alpha, src)    # voice 2: own interpretation
        x = x + m_out
        cache['layers'].append((a_cache, m_cache))
        if track_contrib:
            contrib[f'layer{li}_attn'] = a_out.copy()
            contrib[f'layer{li}_mlp'] = m_out.copy()
    xf, lnf_cache = layernorm_fwd(x)                             # final re-level
    xf_gated = xf * g_embd_fwd[None, :]                          # checked ONCE more
    logits = xf_gated @ P.lm_head
    cache.update(g_embd_bwd=g_embd_bwd, xf_gated=xf_gated, lnf_cache=lnf_cache)
    return (logits, cache, contrib) if track_contrib else (logits, cache)

def backward(P, cache, dlogits, char_idx):
    grads = dict(wte=np.zeros_like(P.wte), wpe=np.zeros_like(P.wpe), lm_head=np.zeros_like(P.lm_head),
                 Wqkv=[np.zeros_like(w) for w in P.Wqkv], Wo=[np.zeros_like(w) for w in P.Wo],
                 Win=[np.zeros_like(w) for w in P.Win], Wout=[np.zeros_like(w) for w in P.Wout])
    d_xf = (dlogits @ P.lm_head.T) * cache['g_embd_bwd'][None, :]
    grads['lm_head'] += cache['xf_gated'].T @ dlogits
    dx = layernorm_bwd(d_xf, cache['lnf_cache'])
    for li in reversed(range(N_LAYER)):
        a_cache, m_cache = cache['layers'][li]
        d_x_mlp, dWin, dWout = mlp_branch_bwd(dx, P, li, m_cache)
        grads['Win'][li] += dWin
        grads['Wout'][li] += dWout
        dx = dx + d_x_mlp
        d_x_attn, dWqkv, dWo = attn_branch_bwd(dx, P, li, a_cache)
        grads['Wqkv'][li] += dWqkv
        grads['Wo'][li] += dWo
        dx = dx + d_x_attn
    d_x0 = dx * cache['g_embd_bwd'][None, :]
    np.add.at(grads['wte'], char_idx, d_x0)
    grads['wpe'][:len(char_idx)] += d_x0
    return grads

print("forward/backward for the full two-voice, two-layer ensemble defined.")

forward/backward for the full two-voice, two-layer ensemble defined.


## Step 9 — rehearsing the ensemble

To rehearse we are going to build the Cat/Dog/Bird corpora, draw a random set of mic levels (`alpha`) from a Dirichlet distribution each rehearsal round — concentrated near the corners, so single-section players get plenty of clean, unblended practice — pick which section's passage actually gets rehearsed based on those mic levels, and correct the arrangement a little each round.

In [12]:
WORDS = {"cat": ["meow", "purr"], "dog": ["woof", "bark"], "bird": ["squawk", "chirp"]}

def make_corpus(words, n_words, generator):
    return " ".join(generator.choice(words) for _ in range(n_words))

corpus_rng = np.random.default_rng(0)
corpora = [make_corpus(WORDS[name], 400, corpus_rng) for name in SOURCE_NAMES]
vocab = sorted(set("".join(corpora)))
V = len(vocab)
c2i = {ch: i for i, ch in enumerate(vocab)}
i2c = {i: ch for ch, i in c2i.items()}

def make_pairs(text):
    ids = [c2i[ch] for ch in text]
    return list(zip(ids[:-1], ids[1:]))

pairs = [make_pairs(t) for t in corpora]
print(f"notes ({V} chars): {vocab}")
print("rehearsal material per section:", {n: len(p) for n, p in zip(SOURCE_NAMES, pairs)})

# rebuild charts now that V is final, and the real model this time
rng = np.random.default_rng(0)
M_embd = make_labels(N_EMBD, K, 0.6, rng)
M_head = [make_labels(N_HEAD, K, 0.6, rng) for _ in range(N_LAYER)]
M_inner = [make_labels(N_INNER, K, 0.6, rng) for _ in range(N_LAYER)]
P = Params(np.random.default_rng(1))

notes (17 chars): [' ', 'a', 'b', 'c', 'e', 'f', 'h', 'i', 'k', 'm', 'o', 'p', 'q', 'r', 's', 'u', 'w']
rehearsal material per section: {'cat': 1998, 'dog': 1998, 'bird': 2593}


In [13]:
LR, N_ITERS, CHUNK = 0.10, 45000, 8
train_rng = np.random.default_rng(123)
losses = []
for it in range(N_ITERS):
    alpha = train_rng.dirichlet([0.3] * K)          # today's three mic levels, mostly near a corner
    src = train_rng.choice(K, p=alpha)              # which section's passage gets rehearsed this round
    p = pairs[src]
    start = train_rng.integers(0, len(p) - CHUNK)
    chunk = p[start:start + CHUNK]
    char_idx = np.array([a for a, b in chunk])
    target_idx = np.array([b for a, b in chunk])

    logits, cache = forward(P, char_idx, alpha, src=src)
    probs = softmax(logits)
    T = len(target_idx)
    loss = -np.log(probs[np.arange(T), target_idx] + 1e-12).mean()
    losses.append(loss)
    dlogits = probs.copy(); dlogits[np.arange(T), target_idx] -= 1.0; dlogits /= T
    grads = backward(P, cache, dlogits, char_idx)

    lr = LR * (1.0 - 0.85 * it / N_ITERS)            # attention needs a lower late-training rate to settle
    P.wte -= lr * grads['wte']
    P.wpe -= lr * grads['wpe']
    P.lm_head -= lr * grads['lm_head']
    for lst, name in [(P.Wqkv, 'Wqkv'), (P.Wo, 'Wo'), (P.Win, 'Win'), (P.Wout, 'Wout')]:
        for w, g in zip(lst, grads[name]):
            w -= lr * g
    if (it + 1) % 5000 == 0:
        print(f"round {it+1:6d}   avg loss (last 500) = {np.mean(losses[-500:]):.3f}")

round   5000   avg loss (last 500) = 1.229
round  10000   avg loss (last 500) = 1.010
round  15000   avg loss (last 500) = 0.985
round  20000   avg loss (last 500) = 0.921
round  25000   avg loss (last 500) = 0.915
round  30000   avg loss (last 500) = 1.039
round  35000   avg loss (last 500) = 0.886
round  40000   avg loss (last 500) = 0.935
round  45000   avg loss (last 500) = 0.932


## Step 10 — Mic Check: does each section still sound like itself?

Pull one mic all the way up and the other two all the way down — a pure corner — and listen to what the ensemble plays. If both voices are wired correctly, each corner should still sound like nothing but that one section.

`generate` truncates its own listening history to the last `BLOCK_SIZE` characters, since that's as far back as attention was ever trained to look. It's also where `temperature` (how strongly the performer favors the single most-expected next note over a less-likely one) and `seed` (which take this is — same mic levels, same temperature, a different roll of the dice unless the seed is fixed) come back into the picture.

In [15]:
def generate(P, alpha, length=60, start_char=" ", seed=0, temperature=0.8):
    g = np.random.default_rng(seed)
    history = [c2i[start_char]]
    out = [start_char]
    for _ in range(length):
        ctx = np.array(history[-BLOCK_SIZE:])
        logits, _ = forward(P, ctx, alpha, src=None)
        probs = softmax(logits[-1] / temperature)
        nxt = g.choice(V, p=probs)
        history.append(nxt)
        out.append(i2c[nxt])
    return "".join(out)

for i, name in enumerate(SOURCE_NAMES):
    print(f"{name:6s} {generate(P, np.eye(K)[i], seed=1)!r}")

cat    ' purr meow pur meow meowr pur pur purhwrrr pur meow pmr meow '
dog    ' f bark baark woof bark wooof wof woof bark bark bark wf wooo'
bird   ' irphirp p squrp chirp eurp c squhqur p chquawk squrk squrk c'


**Note** Keep in mind this is a small corpora so there is a bit of roughness around the edges as there is not a lot of unique information to sample.

## Step 11 — Pulling every section's mic back together

To whether alpha remains independent per mic let's set every section's mic to
full — `(1, 1, 1)` — and then to a quarter — `(0.25, 0.25, 0.25)` — and generate from both. Both settings should keep the *same balance*: no section is favored over the others in either case, only how far all three mics are moved up changes. If the mics only checked anyone once, the pulled-back setting would just be a quieter reading of the same three-section performance.

**What to notice:** However, neither take is a clean phrase — every mic at full means
all three sections are genuinely competing for every note, which is crowded
by nature. But they're not simply the same crowded sound at two volumes either. The dim take should read *less* like a confident three-way performance and more like the ensemble has lost its grip on which section's line it's even trying to play.

Here's the number that explains why: split the arrangement's final contents back into what came from the raw cue (checked once, back in Step 5) versus what came from *both* voices of *both* passes (checked twice each), and compare their sizes at both mic settings.

In [16]:
print("all three mics at full   :", repr(generate(P, np.array([1.0, 1.0, 1.0]), seed=2)))
print("all three mics at a quarter:", repr(generate(P, np.array([0.25, 0.25, 0.25]), seed=2)))

all three mics at full   : ' chwourk purk purrk purrk squaourk puaurk purrk chirrrrrrrrrr'
all three mics at a quarter: ' chwbquafqumchqqusqqqqurrkuauaihukhque squeoumeoqibuawrpurrpi'


In [17]:
def contribution_split(P, text, alpha):
    idx = np.array([c2i[ch] for ch in text])
    _, _, contrib = forward(P, idx, alpha, track_contrib=True)
    cue_norm = np.linalg.norm(contrib['cue'][-1])
    ensemble_norm = sum(np.linalg.norm(v[-1]) for k, v in contrib.items() if k != 'cue')
    return cue_norm, ensemble_norm

probe_text = " meow bark chirp"
e1, d1 = contribution_split(P, probe_text, np.full(K, 1.0))
e0, d0 = contribution_split(P, probe_text, np.full(K, 0.25))

print(f"all mics at full     :  raw-cue contribution={e1:.3f}   ensemble contribution={d1:.3f}")
print(f"all mics at a quarter:  raw-cue contribution={e0:.3f}   ensemble contribution={d0:.3f}")
print(f"\nraw cue dimmed by      {e1/e0:.2f}x   (checked once -- matches the dial ratio exactly)")
print(f"ensemble dimmed by     {d1/d0:.2f}x   (checked twice -- dims much faster)")
print(f"\n=> the raw cue's share of what you're actually hearing is "
      f"{(e0/(e0+d0))/(e1/(e1+d1)):.2f}x larger at the dim setting.")

all mics at full     :  raw-cue contribution=29.615   ensemble contribution=6435.456
all mics at a quarter:  raw-cue contribution=7.404   ensemble contribution=402.115

raw cue dimmed by      4.00x   (checked once -- matches the dial ratio exactly)
ensemble dimmed by     16.00x   (checked twice -- dims much faster)

=> the raw cue's share of what you're actually hearing is 3.95x larger at the dim setting.


**The answer:** pulling every section's mic back together doesn't fade the whole ensemble evenly. The raw cue's contribution dims by exactly the dial ratio (checked once). The ensemble's contribution — both voices, both passes, everything that actually learned what Cat, Dog, and Bird sound like — dims by roughly the *square* of that ratio, because every bit of it passes the mics twice on its way into the arrangement. Pulling every mic back evenly doesn't mean "the same balanced performance, quieter." It means "leaning much more on the raw, un-arranged cue and much less on anything the ensemble actually rehearsed" — which is exactly why the
quarter-mic take above reads as less coherent, not just quieter.

---
## Closing notes

- **Where the real thing lives:** `labs/abcGPT-main/gated_gpt_tent.py`. Its
  attention block checks the mics twice (`M_head` then `M_embd`, before and
  after `c_proj`) exactly the way Step 6 does here; its MLP block checks
  twice the same way (`M_inner` then `M_embd`), exactly like Step 7. This
  notebook's two-voice pass is a direct, full-scope mirror of one of its
  blocks now, not a simplified stand-in for one.
- **No learned scale/shift in the re-leveling step here.** Real
  implementations often add a learned `gamma`/`beta` after normalizing. It
  doesn't change anything in this notebook's argument — those parameters
  apply *after* the scale-cancelling step that matters, so leaving them out
  keeps the code shorter without changing the lesson.

---
## Write the Trained Parameters to Disc
To use the model we just trained above in our frontend we will need to write the trained weights in a place the frontend can access.

In [21]:
import os
os.makedirs("2026-09-abcgpt-spider-graph-frontend/app", exist_ok=True)

save_kwargs = dict(
    wte=P.wte, wpe=P.wpe, lm_head=P.lm_head,
    M_embd=M_embd,
    vocab=np.array(vocab), source_names=np.array(SOURCE_NAMES),
)
for li in range(N_LAYER):
    save_kwargs[f"Wqkv_{li}"] = P.Wqkv[li]
    save_kwargs[f"Wo_{li}"] = P.Wo[li]
    save_kwargs[f"Win_{li}"] = P.Win[li]
    save_kwargs[f"Wout_{li}"] = P.Wout[li]
    save_kwargs[f"Mhead_{li}"] = M_head[li]
    save_kwargs[f"Minner_{li}"] = M_inner[li]

np.savez("2026-09-abcgpt-spider-graph-frontend/app/weights.npz", **save_kwargs)
print("saved", len(save_kwargs), "arrays")

saved 18 arrays


### Check Weights Are Able to Be Pulled

In [18]:
loaded = np.load("app/weights.npz")
VOCAB_L = list(loaded["vocab"]); SRC_L = list(loaded["source_names"])
C2I_L = {ch: i for i, ch in enumerate(VOCAB_L)}
I2C_L = {i: ch for ch, i in C2I_L.items()}
V_L, N_EMBD_L = loaded["wte"].shape
BLOCK_SIZE_L, _ = loaded["wpe"].shape
K_L = len(SRC_L)

N_LAYER = sum (1 for k in _w.files if k.startswith("Wqkv_"))

wte_L, wpe_L, lm_head_L = loaded["wte"], loaded["wpe"], loaded["lm_head"]
M_embd_L = loaded["M_embd"]
Wqkv_L = [loaded[f"Wqkv_{li}"] for li in range(N_LAYER_L)]
Wo_L = [loaded[f"Wo_{li}"] for li in range(N_LAYER_L)]
Win_L = [loaded[f"Win_{li}"] for li in range(N_LAYER_L)]
Wout_L = [loaded[f"Wout_{li}"] for li in range(N_LAYER_L)]
M_head_L = [loaded[f"Mhead_{li}"] for li in range(N_LAYER_L)]
M_inner_L = [loaded[f"Minner_{li}"] for li in range(N_LAYER_L)]
N_HEAD_L = M_head_L[0].shape[0]
HEAD_DIM_L = N_EMBD_L // N_HEAD_L

def layernorm_L(x):
    mu = x.mean(-1, keepdims=True)
    xc = x - mu
    var = (xc ** 2).mean(-1, keepdims=True)
    return xc / np.sqrt(var + EPS)

def gate_L(alpha, M):
    return M @ np.asarray(alpha, dtype=float)

def attn_branch_L(x, li, alpha):
    T = x.shape[0]
    n = layernorm_L(x)
    qkv = n @ Wqkv_L[li]
    q, k, v = np.split(qkv, 3, axis=-1)
    def to_heads(z): return z.reshape(T, N_HEAD_L, HEAD_DIM_L).transpose(1, 0, 2)
    qh, kh, vh = to_heads(q), to_heads(k), to_heads(v)
    scores = (qh @ kh.transpose(0, 2, 1)) / np.sqrt(HEAD_DIM_L)
    mask = np.triu(np.ones((T, T)), k=1).astype(bool)
    scores = np.where(mask[None, :, :], -1e9, scores)
    att = softmax(scores)
    y = (att @ vh) * gate_L(alpha, M_head_L[li])[:, None, None]
    y_flat = y.transpose(1, 0, 2).reshape(T, N_EMBD_L)
    return (y_flat @ Wo_L[li]) * gate_L(alpha, M_embd_L)[None, :]

def mlp_branch_L(x, li, alpha):
    n = layernorm_L(x)
    h, _ = gelu_fwd(n @ Win_L[li])
    h = h * gate_L(alpha, M_inner_L[li])[None, :]
    return (h @ Wout_L[li]) * gate_L(alpha, M_embd_L)[None, :]

def generate_L(alpha, length=60, start_char=" ", seed=0, temperature=0.8):
    g = np.random.default_rng(seed)
    history = [C2I_L[start_char]]
    out = [start_char]
    for _ in range(length):
        ctx = np.array(history[-BLOCK_SIZE_L:])
        g_embd = gate_L(alpha, M_embd_L)
        x = (wte_L[ctx] + wpe_L[:len(ctx)]) * g_embd[None, :]
        for li in range(N_LAYER_L):
            x = x + attn_branch_L(x, li, alpha)
            x = x + mlp_branch_L(x, li, alpha)
        logits = (layernorm_L(x) * g_embd[None, :]) @ lm_head_L
        probs = softmax(logits[-1] / temperature)
        nxt = g.choice(V_L, p=probs)
        history.append(nxt)
        out.append(I2C_L[nxt])
    return "".join(out)

for i, name in enumerate(SOURCE_NAMES):
    orig, reloaded = generate(P, np.eye(K)[i], seed=1), generate_L(np.eye(K)[i], seed=1)
    print(f"{name:6s} {'MATCH' if orig == reloaded else 'MISMATCH'}")

NameError: name '_w' is not defined